In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [26]:
# clear everything
SAMPLES_PATH = Path.cwd().parent / 'samples'

for path in SAMPLES_PATH.rglob('*'):
    if path.is_file() and path.name.endswith('.npy'):
        path.unlink()

## Samples from online scraping

In [27]:
BASE_PATH = Path.cwd().parent / 'data' / 'online-scraping' 
SAVE_PATH = Path.cwd().parent / 'samples' / 'positive'

SAVE_PATH.mkdir(parents=True, exist_ok=True)
assert BASE_PATH.exists() and SAVE_PATH.exists()

In [28]:
from scipy.interpolate import interp1d
from scipy.signal import butter, filtfilt

def bandpass_filter(signal, sample_rate=1000, bandpass_low=0.5, bandpass_high=150):
    """
    Apply a band-pass filter.
    """

    # nyquist frequency
    nyquist = sample_rate / 2
    
    # band-pass filter
    b_bp, a_bp = butter(4, [bandpass_low/nyquist, bandpass_high/nyquist], btype='band')
    return filtfilt(b_bp, a_bp, signal)

def resample(raw_lead: list[tuple[float, float]], sample_rate = 1000) -> list[float]:
    """
    Convert a list of (time, voltage) coordinates into 1000 Hz ECG. 
    """
    
    # unzip pairs into two lists
    timestamps, voltages = zip(*raw_lead)
    timestamps = np.array(timestamps)
    voltages = np.array(voltages)
    
    # remove possible duplicates
    unique_timestamps, indices = np.unique(timestamps, return_index=True)
    unique_voltages = voltages[indices]

    # must be strictly increasing
    assert np.all(np.diff(unique_timestamps) > 0)
        
    start_time = unique_timestamps[0]
    end_time = unique_timestamps[-1]

    interpolate_function = interp1d(
        unique_timestamps, 
        unique_voltages, 
        kind='linear', 
        fill_value='extrapolate'
    )
    
    uniform_timestamps = np.arange(start_time, end_time, 1/sample_rate)
    uniform_voltages = interpolate_function(uniform_timestamps)
    
    return uniform_voltages

In [29]:
for path in BASE_PATH.iterdir():
    filename = path.name
    print(f'processing {filename}')
    try:
        signal = np.load(path, allow_pickle=True)
        v1, v2, v3 = signal
        v1 = resample(v1)
        v2 = resample(v2)
        v3 = resample(v3)

        min_length = min(len(v1), len(v2), len(v3))
        v1 = v1[:min_length]
        v2 = v2[:min_length]
        v3 = v3[:min_length]

        # Apply filters after resampling
        v1 = bandpass_filter(v1)
        v2 = bandpass_filter(v2)
        v3 = bandpass_filter(v3)
        resampled = [v1, v2, v3]

        np.save(SAVE_PATH / f'positive_{filename}', resampled)
        print(f'saved {filename}')
        
    except Exception as e:
        print(f'error processing {filename}: {e}')
        continue

file_count = len([*(SAVE_PATH).iterdir()])
f'saved {file_count} files to {SAVE_PATH}' 

processing 027.npy
saved 027.npy
processing 028.npy
saved 028.npy
processing 040.npy
saved 040.npy
processing 022.npy
saved 022.npy
processing 049.npy
saved 049.npy
processing 016.npy
saved 016.npy
processing 007.npy
saved 007.npy
processing 025.npy
saved 025.npy
processing 043.npy
saved 043.npy
processing 033.npy
saved 033.npy
processing 020.npy
saved 020.npy
processing 037.npy
saved 037.npy
processing 045.npy
saved 045.npy
processing 021.npy
saved 021.npy
processing 041.npy
saved 041.npy
processing 026.npy
saved 026.npy
processing 004.npy
saved 004.npy
processing 015.npy
saved 015.npy
processing 044.npy
saved 044.npy
processing 008.npy
saved 008.npy
processing 023.npy
saved 023.npy
processing 001.npy
saved 001.npy
processing 029.npy
saved 029.npy
processing 014.npy
saved 014.npy
processing 002.npy
saved 002.npy
processing 003.npy
saved 003.npy
processing 013.npy
saved 013.npy
processing 000.npy
saved 000.npy
processing 010.npy
saved 010.npy
processing 036.npy
saved 036.npy
processing

'saved 33 files to /home/elijah/CS19X-Brugada-Syndrome-Classifier/samples/positive'

## Samples from PTB-XL

In [30]:
import ast 
import wfdb

BASE_PATH = Path.cwd().parent / 'data' / 'ptbxl' 
SAVE_PATH = Path.cwd().parent / 'samples' / 'negative'

SAVE_PATH.mkdir(parents=True, exist_ok=True)
assert BASE_PATH.exists() and SAVE_PATH.exists()

In [31]:
metadata = pd.read_csv(BASE_PATH / 'ptbxl_database.csv')
f'Found {len(metadata)} ECG entries'

'Found 21799 ECG entries'

In [32]:
# make sure codes say "normal"
def is_normal(raw_scp_codes):
    codes = ast.literal_eval(raw_scp_codes) 
    return codes.get('NORM') == 100.0

def is_brugada_like(raw_scp_codes):
    codes = ast.literal_eval(raw_scp_codes)
    brugada_codes = [
        'STE_',
        'IRBBB',
        'CRBBB',
        'INVT',
        'NST', 
        'NDT',
    ]

    return any([code in codes for code in brugada_codes])


def is_rbbb(raw_scp_codes):
    codes = ast.literal_eval(raw_scp_codes)
    return any(key in codes for key in ['IRBBB', 'CRBBB'])
    
def shortlist(metadata, criteria, start: int | None, end: int | None, exclude=None):
    """Return top-n rows matching `criteria` AND having no artifacts; optionally exclude filenames."""

    if criteria: # e.g. criteria=is_brugada_like
        candidates = metadata[metadata['scp_codes'].apply(criteria)]
    else:
        candidates = metadata

    if exclude:
        candidates = candidates[~candidates['filename_hr'].isin(exclude)]

    clean = candidates[candidates.apply(is_clean_row, axis=1)]

    if start is not None or end is not None:
        return clean.iloc[start:end]
    else:
        return clean

def is_empty_field(val):
    """True when a metadata field should be considered empty/None."""
    if pd.isna(val):
        return True
    if isinstance(val, (bool, np.bool_)):
        return not val
    if isinstance(val, (int, float, np.integer, np.floating)):
        return val == 0 or pd.isna(val)
    s = str(val).strip().lower()
    return s in ('', 'none', 'nan', 'false', '0', '0.0', 'unknown')

def is_clean_row(row):
    for col in [
        'baseline_drift', 
        'static_noise', 
        'burst_noise',
        'electrodes_problems', 
        'extra_beats', 
        'pacemaker', 
        'heart_axis',
    ]:
        if col not in row:
            continue
        if not is_empty_field(row[col]):
            return False
    return True

N = 50 
BAD_SAMPLES = [
    'records500/00000/00088_hr' # contains a heartbeat 10mm higher than the rest
]

negative_shortlist = shortlist(metadata, criteria=None, start=0, end=50, exclude=BAD_SAMPLES)

len(negative_shortlist)

50

In [33]:
negative_shortlist['scp_codes']

1                   {'NORM': 80.0, 'SBRAD': 0.0}
2                     {'NORM': 100.0, 'SR': 0.0}
9                     {'NORM': 100.0, 'SR': 0.0}
11                  {'NORM': 80.0, 'SBRAD': 0.0}
13                    {'NORM': 100.0, 'SR': 0.0}
18                    {'NORM': 100.0, 'SR': 0.0}
20                    {'NORM': 100.0, 'SR': 0.0}
22                               {'AFLT': 100.0}
23                    {'NORM': 100.0, 'SR': 0.0}
28                    {'NORM': 100.0, 'SR': 0.0}
30                    {'NORM': 100.0, 'SR': 0.0}
32                    {'NORM': 100.0, 'SR': 0.0}
34                    {'NORM': 100.0, 'SR': 0.0}
36                  {'NORM': 80.0, 'SARRH': 0.0}
41                    {'NORM': 100.0, 'SR': 0.0}
42                    {'NORM': 100.0, 'SR': 0.0}
43                    {'NORM': 100.0, 'SR': 0.0}
45                    {'NORM': 100.0, 'SR': 0.0}
51                   {'IRBBB': 100.0, 'SR': 0.0}
52                    {'NORM': 100.0, 'SR': 0.0}
53                  

In [34]:
def load_signal(filename):
    path = BASE_PATH / filename
    path = str(path).replace('.dat', '').replace('.hea', '')
    signal = wfdb.rdrecord(path)
    return signal

def get_index(filename):
    *parents, name = filename.split("/")
    index, resolution = name.split("_")
    return int(index)

desired_leads = [f'V{i}' for i in range(1, 7)]

for filename in negative_shortlist['filename_hr']:
    index = get_index(filename)
    print(f'processing {index:05d}')
    
    try:
        signal = load_signal(filename)
    except Exception as e:
        print(f"Error loading {filename}: {e}")
        continue

    leads_dict = {name: signal.p_signal[:, j] for j, name in enumerate(signal.sig_name)}
    leads = []
    for lead_name in desired_leads:
        lead = leads_dict.get(lead_name)
        if lead is None:
            print(f'Skip {filename}: missing {lead_name}')
            leads = []
            break
        leads.append(lead)

    if not leads:
        continue

    if signal.fs != 1000:
        scale_factor = 1000 / signal.fs
        leads = [np.interp(np.arange(0, len(lead), 1/scale_factor), np.arange(len(lead)), lead) for lead in leads]

    leads = [bandpass_filter(lead) for lead in leads]
    sample = {'lead_names': desired_leads, 'signal': np.stack(leads)}
    out_name = f'negative_{index:03d}.npy'
    np.save(SAVE_PATH / out_name, sample)
    print(f'saved to {out_name}')

file_count = len([*SAVE_PATH.iterdir()])
f'saved {file_count} files to {SAVE_PATH}'


processing 00002
saved to negative_002.npy
processing 00003
saved to negative_003.npy
processing 00010
saved to negative_010.npy
processing 00012
saved to negative_012.npy
processing 00014
saved to negative_014.npy
processing 00019
saved to negative_019.npy
processing 00021
saved to negative_021.npy
processing 00023
saved to negative_023.npy
processing 00024
saved to negative_024.npy
processing 00029
saved to negative_029.npy
processing 00031
saved to negative_031.npy
processing 00033
saved to negative_033.npy
processing 00035
saved to negative_035.npy
processing 00037
saved to negative_037.npy
processing 00042
saved to negative_042.npy
processing 00043
saved to negative_043.npy
processing 00044
saved to negative_044.npy
processing 00046
saved to negative_046.npy
processing 00052
saved to negative_052.npy
processing 00053
saved to negative_053.npy
processing 00054
saved to negative_054.npy
processing 00055
saved to negative_055.npy
processing 00057
saved to negative_057.npy
processing 

'saved 50 files to /home/elijah/CS19X-Brugada-Syndrome-Classifier/samples/negative'

# Brugada HUCA: Positive

# Brugada HUCA: Negative

# Samples from LIFECARE

In [35]:
BASE_PATH = Path.cwd().parent / 'data' / 'lifecare'
SAVE_PATH = Path.cwd().parent / 'samples' / 'positive-lifecare'

SAVE_PATH.mkdir(parents=True, exist_ok=True)
BASE_PATH.exists() and SAVE_PATH.exists()

True

In [36]:
for path in BASE_PATH.iterdir():
    filename = path.name
    print(f'processing {filename}')
    try:
        signal = np.load(path, allow_pickle=True)
        v1, v2, v3 = signal
        v1 = resample(v1)
        v2 = resample(v2)
        v3 = resample(v3)

        min_length = min(len(v1), len(v2), len(v3))
        v1 = v1[:min_length]
        v2 = v2[:min_length]
        v3 = v3[:min_length]

        # Apply filters after resampling
        v1 = bandpass_filter(v1)
        v2 = bandpass_filter(v2)
        v3 = bandpass_filter(v3)
        resampled = [v1, v2, v3]

        np.save(SAVE_PATH / f'positive_{filename}', resampled)
        print(f'saved {filename}')
        
    except Exception as e:
        print(f'error processing {filename}: {e}')
        continue

file_count = len([*(SAVE_PATH).iterdir()])
f'saved {file_count} files to {SAVE_PATH}' 

processing 061.npy
saved 061.npy
processing 053.npy
saved 053.npy
processing 058.npy
saved 058.npy
processing 054.npy
saved 054.npy
processing 051.npy
saved 051.npy
processing 052.npy
saved 052.npy
processing 055.npy
saved 055.npy
processing 060.npy
saved 060.npy
processing 062.npy
saved 062.npy
processing 059.npy
saved 059.npy


'saved 10 files to /home/elijah/CS19X-Brugada-Syndrome-Classifier/samples/positive-lifecare'